# H-Bar Symbolic - Kaggle Execution Notebook (Vectorized)

This notebook executes the H-Bar pilot study (N=15) on Kaggle GPUs using the **vectorized training engine** for maximum efficiency.

## Key Optimizations

1. **jax.vmap across N runs**: Train 15 models simultaneously on single GPU
2. **Early stopping**: Stop when σ̃_A > 0.90 (crystallization detection)
3. **Mixed precision**: bfloat16 activations with float32 parameters for 2x speedup

## Objectives

1. **Verify H-Bar Effect:** Confirm OOD accuracy >90% and σ̂_A >0.9
2. **Compare Conditions:** Test both Additive (B) and Multiplicative (C) loss coupling
3. **Check Stability:** Ensure training doesn't diverge (especially for multiplicative)
4. **Phase 2 Detection:** Verify crystallization occurs within first 1,000 steps

## Expected Outcomes

| Metric | Baseline | H-Bar Target |
|--------|----------|--------------|
| ID Accuracy | 91.9% | >90% |
| OOD Accuracy | 63.0% | >90% |
| σ̂_A | 0.685 | >0.9 |
| GCA (g_A) | -0.02 | Positive |
| Phase 2 Entry | Never | <1,000 steps |

## Estimated Runtime

- **Sequential (old):** ~8 hours for N=15
- **Vectorized (new):** ~20-30 minutes for N=15 ✅
- Kaggle session limit: 12 hours ✓

## Step 1: Environment Setup

First, we'll clone the repository and install dependencies.

In [ ]:
# Clone the repository
!git clone https://github.com/basyirin-dev/hbar-symbolic.git
%cd hbar-symbolic

# List files to verify
!ls -la

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

# Install the package
!pip install -q -e .

In [ ]:
# Verify JAX installation and GPU availability
import jax
import jax.numpy as jnp

print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")
print(f"Backend: {jax.default_backend()}")

## Step 2: Mount Baseline Dataset

Mount the baseline results dataset for comparison with H-Bar trajectories.

In [ ]:
# Check for baseline dataset
!ls -la /kaggle/input/datasets/basyirinamsyar/baseline-metrics-and-model-params/

# Copy baseline files for comparison
!cp /kaggle/input/datasets/basyirinamsyar/baseline-metrics-and-model-params/baseline_metrics.csv ./baseline_metrics.csv 2>/dev/null || echo 'Baseline metrics not found, using existing'
!cp /kaggle/input/datasets/basyirinamsyar/baseline-metrics-and-model-params/model_params.msgpack ./model_params.msgpack 2>/dev/null || echo 'Model params not found, using existing'

print("Baseline files copied successfully")

## Step 3: Verify Data Files

Ensure the frozen evaluation datasets are present.

In [ ]:
# Check data directory
!ls -la data/

# Verify file sizes
import os
for f in ['scan_id_eval.json', 'scan_ood_eval.json', 'cogs_id_eval.json', 'cogs_ood_eval.json']:
    path = os.path.join('data', f)
    if os.path.exists(path):
        size = os.path.getsize(path)
        print(f"{f}: {size:,} bytes")
    else:
        print(f"WARNING: {f} not found!")

## Step 4: Run Pilot Study - Condition C (Multiplicative)

Execute N=15 runs with multiplicative loss coupling using the **vectorized training engine** (~30 minutes).

In [ ]:
# Run multiplicative condition (Condition C) - 15 runs (VECTORIZED)
!python scripts/train_hbar_vectorized.py \
    --domain scan \
    --condition multiplicative \
    --n_runs 15 \
    --batch_size 64 \
    --total_steps 5000 \
    --learning_rate 0.001 \
    --lambda_sigma 0.5 \
    --base_seed 42 \
    --output_dir .

## Step 5: Run Pilot Study - Condition B (Additive)

Execute N=15 runs with additive loss coupling using the **vectorized training engine** (~30 minutes).

In [ ]:
# Run additive condition (Condition B) - 15 runs (VECTORIZED)
!python scripts/train_hbar_vectorized.py \
    --domain scan \
    --condition additive \
    --n_runs 15 \
    --batch_size 64 \
    --total_steps 5000 \
    --learning_rate 0.001 \
    --lambda_sigma 0.5 \
    --base_seed 42 \
    --output_dir .

## Step 6: Analyze Pilot Results

Load and analyze the pilot results to verify the H-Bar effect.

In [ ]:
import pandas as pd
import numpy as np

# Load pilot summary
summary = pd.read_csv('pilot_multiplicative_summary.csv')
print("Pilot Results Summary (Multiplicative):")
print(summary.head(10))

In [ ]:
# Load additive summary
summary_additive = pd.read_csv('pilot_additive_summary.csv')
print("Pilot Results Summary (Additive):")
print(summary_additive.head(10))

In [ ]:
# Compute statistics by condition
print("\n" + "="*60)
print("PILOT STUDY RESULTS BY CONDITION")
print("="*60)

for condition, summary in [('multiplicative', pd.read_csv('pilot_multiplicative_summary.csv')), 
                           ('additive', pd.read_csv('pilot_additive_summary.csv'))]:
    print(f"\n--- {condition.upper()} Condition (N={len(summary)}) ---")
    
    # Crystallization stats
    crystallized = summary['crystallized'].sum()
    print(f"Crystallized (σ̃_A > 0.90): {crystallized}/{len(summary)} runs")
    
    crystallized_steps = summary[summary['crystallized'] == True]['crystallization_step']
    if len(crystallized_steps) > 0:
        print(f"Mean crystallization step: {crystallized_steps.mean():.1f}")
        print(f"Min crystallization step: {crystallized_steps.min()}")
        print(f"Max crystallization step: {crystallized_steps.max()}")

## Step 7: Visualize Training Metrics

Plot the training metrics from the CSV logs to observe crystallization dynamics.

In [ ]:
import matplotlib.pyplot as plt

# Load vectorized training metrics
metrics = pd.read_csv('hbar_multiplicative_vectorized_metrics.csv')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('H-Bar Vectorized Training Trajectories', fontsize=16)

# Plot Loss per run
ax = axes[0, 0]
for run_id in metrics['run_id'].unique()[:5]:  # Plot first 5 runs
    run_data = metrics[metrics['run_id'] == run_id]
    ax.plot(run_data['step'], run_data['train_loss'], alpha=0.5, label=f'Run {run_id}')
ax.set_title('Training Loss (Multiplicative)')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.legend(fontsize=8)

# Plot σ̃_A per run
ax = axes[0, 1]
for run_id in metrics['run_id'].unique()[:5]:
    run_data = metrics[metrics['run_id'] == run_id]
    ax.plot(run_data['step'], run_data['sigma_tilde'], alpha=0.5, label=f'Run {run_id}')
ax.axhline(y=0.9, color='green', linestyle='--', label='Crystallization threshold')
ax.set_title('σ̃_A Trajectory (Multiplicative)')
ax.set_xlabel('Step')
ax.set_ylabel('σ̃_A')
ax.legend(fontsize=8)

# Plot ID Loss
ax = axes[1, 0]
for run_id in metrics['run_id'].unique()[:5]:
    run_data = metrics[metrics['run_id'] == run_id]
    ax.plot(run_data['step'], run_data['id_loss'], alpha=0.5)
ax.set_title('ID Stream Loss (Multiplicative)')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')

# Plot OOD Loss
ax = axes[1, 1]
for run_id in metrics['run_id'].unique()[:5]:
    run_data = metrics[metrics['run_id'] == run_id]
    ax.plot(run_data['step'], run_data['ood_loss'], alpha=0.5)
ax.set_title('OOD Stream Loss (Multiplicative)')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')

plt.tight_layout()
plt.savefig('hbar_vectorized_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 8: Save Results for Phase 4

Package the results for download and further analysis.

In [ ]:
import shutil

# Create results directory
!mkdir -p hbar_pilot_results

# Copy all CSV files
!cp pilot_*_summary.csv hbar_pilot_results/
!cp hbar_*_vectorized_metrics.csv hbar_pilot_results/

# Copy the training trajectory plot
!cp hbar_vectorized_trajectories.png hbar_pilot_results/ 2>/dev/null || true

print("Saved to: hbar_pilot_results/")

# List all output files
!ls -la hbar_pilot_results/

## Step 9: Next Steps

Based on the pilot results:

### If H-Bar Effect Confirmed (crystallization in most runs):
1. Proceed to full N=120 pre-registered runs
2. Run on both SCAN and COGS domains
3. Submit results for publication

### If H-Bar Effect Not Confirmed:
1. Debug signal extraction pipeline
2. Adjust hyperparameters (λ_σ, κ_α)
3. Re-run pilot with modifications

### Commands for Full N=120 Run (Vectorized):
```bash
# Run in parallel (2 Kaggle sessions)
# Session 1: Runs 1-60
python scripts/train_hbar_vectorized.py --domain scan --condition multiplicative --n_runs 60 --base_seed 42

# Session 2: Runs 61-120
python scripts/train_hbar_vectorized.py --domain scan --condition multiplicative --n_runs 60 --base_seed 60042
```